# SignSage · FDMSE-ISL extraction v2 (all 2,002 words, resumable)

Turns the whole FDMSE-ISL dataset into saved hand/pose/face landmarks, then into model-ready features.
A disconnect never costs more than a few minutes of work.

**Before you start**
- Colab: *Runtime → Change runtime type → **CPU***. This step is CPU-bound; a GPU gives no speed-up and burns your quota. Keep the browser tab open.
- Colab secret: name **`FDMSE_ISL_KEY`**, and switch **Notebook access ON for this notebook** (the switch is per notebook). On a laptop, set the environment variable instead. The key is never printed.

**Cells, in order**
1 Settings · 2 Setup · 3-5 write the code files · **6 Access check** · 7 Archive download (only if cell 6 asks for it) · 8 Initialise · 9 Benchmark (once) · **10 Run extraction** · 11 Status/verify · 12 Build features

**After any disconnect:** run 1 → 2 → 3 → 4 → 5 → 6 → (7) → 8 → 10. Finished chunks are skipped automatically.

**How it works**
- Cell 6 tests the portal and explains any problem in plain words. If range requests work, videos are streamed one by one (no 30 GB download). If the portal refuses ranges but allows normal downloads, cell 7 downloads the archive once, resumably, to Drive.
- Work is cut into chunks of 50 videos; each finished chunk is saved to Drive at once. Raw landmarks are kept so features can be rebuilt without repeating the slow part.
- Two machines can share the job (see the end of the notebook). Settings are locked on the first run and later runs refuse to mix them.

In [ ]:
# ============================ 1. SETTINGS (edit only here) ============================
LOCAL_ARCHIVE = None   # full path of FDMSE-ISL.zip if you ALREADY HAVE it (then no key and no portal are needed)
                       # Colab example: "/content/drive/MyDrive/SignSage_FDMSE_v2/archive/FDMSE-ISL.zip"
ACCESS_MODE   = "auto" # "auto" = cell 6 decides ("stream" or "archive"); or force one of them

MACHINE_INDEX = 0      # which machine this is (Colab = 0, laptop = 1) - only matters if MACHINE_COUNT > 1
MACHINE_COUNT = 1      # 1 = this machine does ALL chunks. 2 = split work with a second machine (modulo split)
ORDER         = "forward"  # "forward" or "reverse" - handy for two machines working towards each other

CHUNK_SIZE    = 50     # videos per saved chunk (do not change after the first run)
N_FRAMES      = 48     # frames sampled per video (do not change after the first run)
FACE_EVERY_K  = 4      # run the slow face model every K-th frame; 1 = every frame (slowest, exact)
POSE_MODEL    = "full" # "full" or "lite"
MAX_SIDE      = 720    # shrink frames so the longest side is at most this (0 = never)

WORKERS       = None   # parallel processes; None = number of CPU cores
MAX_HOURS     = None   # e.g. 3.0 -> stop cleanly after 3 hours (finished chunks are kept)
LIMIT_CHUNKS  = None   # e.g. 2 -> quick trial of only 2 chunks
OUT_ROOT      = None   # None -> Google Drive on Colab, ./signsage_fdmse_v2 elsewhere
DRY_RUN       = False  # True = test the pipeline with a FAKE detector (separate *_dryrun folder)

In [ ]:
# ============================ 2. SETUP ============================
import os, sys, json, getpass, importlib, importlib.metadata, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = OUT_ROOT or "/content/drive/MyDrive/SignSage_FDMSE_v2"
    WORK = "/content/work"
else:
    OUT_ROOT = OUT_ROOT or os.path.abspath("./signsage_fdmse_v2")
    WORK = os.path.abspath("./work")
if DRY_RUN:
    OUT_ROOT = OUT_ROOT.rstrip("/") + "_dryrun"

META, CHUNKS, FEATS = (f"{OUT_ROOT}/{d}" for d in ("meta", "chunks", "features"))
ARCHIVE_PATH = LOCAL_ARCHIVE or f"{OUT_ROOT}/archive/FDMSE-ISL.zip"
if LOCAL_ARCHIVE and not os.path.exists(LOCAL_ARCHIVE):
    raise SystemExit(f"LOCAL_ARCHIVE not found: {LOCAL_ARCHIVE}\nOn Colab, check that Drive is mounted and the path/file name is exact.")
HAVE_ARCHIVE = os.path.exists(ARCHIVE_PATH)                    # an archive you already have = no key, no portal
LOCAL_CHUNKS = f"{WORK}/chunks_local" if IN_COLAB else CHUNKS   # fast local disk first on Colab, then copied to Drive
MODEL_DIR, TMP_DIR = f"{WORK}/models", f"{WORK}/tmp"
for d in (META, CHUNKS, FEATS, LOCAL_CHUNKS, MODEL_DIR, TMP_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.getcwd())

MP_VERSION = ""
if not DRY_RUN:
    try:
        import mediapipe  # noqa
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe"], check=True)
    MP_VERSION = importlib.metadata.version("mediapipe")
    try:
        import cv2  # noqa
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)

def get_key():
    k = ""
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = (userdata.get("FDMSE_ISL_KEY") or "").strip()
        except Exception as e:
            print(f"Colab secret not readable ({e.__class__.__name__}): is 'Notebook access' ON for this notebook?")
    k = k or os.environ.get("FDMSE_ISL_KEY", "").strip()
    if not k:
        print("No key found in the secret or environment - paste it at the hidden prompt.")
        k = getpass.getpass("FDMSE-ISL access key (hidden): ").strip()
    return k

KEY = "" if HAVE_ARCHIVE else get_key()
URL_TEMPLATE = os.environ.get("FDMSE_ISL_URL_TEMPLATE", "https://cs.rkmvu.ac.in/~isl/isl-dataset/file?key={key}")
print(f"colab={IN_COLAB} | mediapipe={MP_VERSION or '(dry run)'} | output -> {OUT_ROOT}")
if HAVE_ARCHIVE:
    print(f"using the archive that is already here: {ARCHIVE_PATH}  (no key or portal needed)")
else:
    print(f"key loaded: {bool(KEY)} ({len(KEY)} characters; contents never shown)")

In [ ]:
%%writefile sp_core.py
"""SignSage FDMSE-ISL extraction core (v2).

Design goals
  * Nothing is ever downloaded as one 30 GB file: videos are streamed from the
    portal with HTTP Range requests, one at a time (the portal supports ranges).
  * Work is split into small deterministic CHUNKS. A finished chunk is written
    atomically, so a disconnect loses at most the chunk being processed.
  * Raw landmarks are saved (not only features) so features can be rebuilt
    later without re-running the slow detectors.
  * The access key is never printed: every error message is redacted.
"""
from __future__ import annotations

import hashlib
import io
import json
import os
import re
import tempfile
import time
import zipfile
from pathlib import Path

import numpy as np

POSE_N, HAND_N, FACE_N = 33, 21, 468


# --------------------------------------------------------------------------
# errors + redaction
# --------------------------------------------------------------------------
class NetworkError(RuntimeError):
    """Transient/IO failure fetching data. The whole chunk is retried later."""


class AuthError(NetworkError):
    """Portal rejected the key (HTTP 401/403). Retrying will not help."""


_SECRETS: list[str] = []


def register_secret(s: str) -> None:
    if s and s not in _SECRETS:
        _SECRETS.append(s)


def redact(text: str) -> str:
    out = str(text)
    for s in _SECRETS:
        out = out.replace(s, "<KEY>")
    return re.sub(r"(key=)[0-9A-Za-z]{8,}", r"\1<KEY>", out)


# --------------------------------------------------------------------------
# remote ZIP access through HTTP Range requests
# --------------------------------------------------------------------------
class RangeFile(io.RawIOBase):
    """Read-only, seekable file object backed by HTTP Range requests.

    Read-ahead buffering is essential: zipfile reads in tiny 4 KB pieces, which
    would otherwise become thousands of HTTP requests per video.
    """

    def __init__(self, url, size=None, readahead=2 << 20, retries=6,
                 timeout=(30, 120), session=None):
        import requests
        self._requests = requests
        self.url = url
        self.readahead = readahead
        self.retries = retries
        self.timeout = timeout
        self.session = session or requests.Session()
        self._pos = 0
        self._buf = b""
        self._buf_start = 0
        self.size = size if size is not None else self._probe_size()

    # -- network helpers ---------------------------------------------------
    def _probe_size(self):
        r = self._request({"Range": "bytes=0-0"}, stream=True)
        try:
            cr = r.headers.get("Content-Range", "")
            if "/" in cr and cr.split("/")[-1].isdigit():
                return int(cr.split("/")[-1])
            raise NetworkError("portal did not report a total size via Content-Range "
                               "(Range requests unsupported?)")
        finally:
            r.close()

    def _request(self, headers, stream=False):
        last = None
        for attempt in range(1, self.retries + 1):
            try:
                r = self.session.get(self.url, headers=headers, timeout=self.timeout,
                                     stream=stream)
                if r.status_code in (401, 403):
                    r.close()
                    raise AuthError(f"portal returned HTTP {r.status_code} (key rejected?)")
                if r.status_code == 206:
                    return r
                if r.status_code == 200:
                    r.close()
                    raise NetworkError("portal ignored the Range header (HTTP 200); "
                                       "refusing to download the whole archive")
                last = f"HTTP {r.status_code}"
                r.close()
            except AuthError:
                raise
            except NetworkError:
                raise
            except Exception as e:  # requests exceptions embed the URL (and key!)
                last = f"{e.__class__.__name__}: {redact(e)}"
            time.sleep(min(2 ** attempt, 30))
        raise NetworkError(f"giving up after {self.retries} attempts ({last})")

    def _get(self, start, end_inclusive):
        r = self._request({"Range": f"bytes={start}-{end_inclusive}"})
        try:
            data = r.content
        except Exception as e:
            raise NetworkError(f"{e.__class__.__name__}: {redact(e)}")
        finally:
            r.close()
        want = end_inclusive - start + 1
        if len(data) != want:
            raise NetworkError(f"short read: wanted {want} bytes, got {len(data)}")
        return data

    # -- file protocol -----------------------------------------------------
    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self._pos

    def seek(self, offset, whence=0):
        if whence == 0:
            new = offset
        elif whence == 1:
            new = self._pos + offset
        elif whence == 2:
            new = self.size + offset
        else:
            raise ValueError("bad whence")
        if new < 0:
            raise ValueError("negative seek position")
        self._pos = new
        return self._pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self._pos
        n = min(n, self.size - self._pos)
        if n <= 0:
            return b""
        end = self._pos + n
        if not (self._buf_start <= self._pos and end <= self._buf_start + len(self._buf)):
            fetch_end = min(self.size, self._pos + max(n, self.readahead))
            self._buf = self._get(self._pos, fetch_end - 1)
            self._buf_start = self._pos
        off = self._pos - self._buf_start
        data = self._buf[off:off + n]
        self._pos += len(data)
        return data

    def readinto(self, b):
        data = self.read(len(b))
        b[:len(data)] = data
        return len(data)

    def prefetch(self, start, length):
        """Load exactly [start, start+length) into the buffer with a single request."""
        end = min(self.size, start + length)
        if end > start:
            self._buf = self._get(start, end - 1)
            self._buf_start = start


def open_remote_zip(url):
    return zipfile.ZipFile(RangeFile(url))


def open_source(source):
    """source = https URL (stream with Range requests) or a local .zip path."""
    if str(source).lower().startswith(("http://", "https://")):
        return open_remote_zip(source)
    return zipfile.ZipFile(str(source))


def read_member(zf, name):
    """Read one archive member. For remote archives the whole member is fetched in a
    single request (no wasted read-ahead, ~1 request per video)."""
    fp = getattr(zf, "fp", None)
    if isinstance(fp, RangeFile):
        info = zf.getinfo(name)
        fp.prefetch(info.header_offset,
                    30 + len(info.filename.encode("utf-8")) + 512 + info.compress_size)
    return zf.read(name)


# --------------------------------------------------------------------------
# frame sampling
# --------------------------------------------------------------------------
def sample_frames(video_path, n, max_side):
    """Return exactly n RGB frames evenly spaced over the clip (repeating frames
    if the clip is shorter than n). Raises ValueError for unreadable videos."""
    import cv2
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError("cannot open video")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total <= 0:
        total = 1
        while cap.grab():
            total += 1
        cap.release()
        cap = cv2.VideoCapture(str(video_path))
    idxs = np.linspace(0, max(total - 1, 0), n).round().astype(int)
    needed = {}
    for pos, i in enumerate(idxs.tolist()):
        needed.setdefault(i, []).append(pos)
    frames = [None] * n
    last_idx = int(idxs.max())
    i = 0
    while i <= last_idx:
        if not cap.grab():
            break
        if i in needed:
            ok, bgr = cap.retrieve()
            if ok:
                h, w = bgr.shape[:2]
                if max_side and max(h, w) > max_side:
                    s = max_side / float(max(h, w))
                    bgr = cv2.resize(bgr, (int(round(w * s)), int(round(h * s))),
                                     interpolation=cv2.INTER_AREA)
                rgb = np.ascontiguousarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
                for pos in needed[i]:
                    frames[pos] = rgb
        i += 1
    cap.release()
    good = [k for k, f in enumerate(frames) if f is not None]
    if not good:
        raise ValueError("no decodable frames")
    for k in range(n):                       # forward fill, then backward fill
        if frames[k] is None:
            prev = [g for g in good if g < k]
            frames[k] = frames[prev[-1]] if prev else frames[good[0]]
    return frames


# --------------------------------------------------------------------------
# detectors (MediaPipe Tasks API; same calls the old notebook already used)
# --------------------------------------------------------------------------
MODEL_URLS = {
    "pose_full": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
    "pose_lite": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
    "hand": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    "face": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
}
MODEL_FILES = {k: Path(v).name for k, v in MODEL_URLS.items()}


def download_models(model_dir, pose_model):
    """Call ONCE from the main process (parallel workers must not race to download)."""
    import urllib.request
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    for name in (f"pose_{pose_model}", "hand", "face"):
        dst = model_dir / MODEL_FILES[name]
        if dst.exists() and dst.stat().st_size > 100_000:
            continue
        tmp = dst.with_suffix(".part")
        urllib.request.urlretrieve(MODEL_URLS[name], tmp)
        os.replace(tmp, dst)
    return model_dir


class Detectors:
    """detect(rgb, do_face) -> (pose (33,3)|None, hands [(21,3,label_code)], face (468,3)|None)
    label_code: 1 = 'Left', 2 = 'Right' (as reported by MediaPipe; v3 ignores labels)."""

    def __init__(self, model_dir, pose_model):
        import mediapipe as mp
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision
        self.mp = mp
        self.timers = {"pose": 0.0, "hand": 0.0, "face": 0.0, "pose_n": 0, "hand_n": 0, "face_n": 0}
        model_dir = Path(model_dir)
        self.pose = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
            base_options=mp_python.BaseOptions(
                model_asset_path=str(model_dir / MODEL_FILES[f"pose_{pose_model}"])),
            num_poses=1, output_segmentation_masks=False))
        self.hand = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
            base_options=mp_python.BaseOptions(
                model_asset_path=str(model_dir / MODEL_FILES["hand"])),
            num_hands=2))
        self.face = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
            base_options=mp_python.BaseOptions(
                model_asset_path=str(model_dir / MODEL_FILES["face"])),
            num_faces=1, output_face_blendshapes=False,
            output_facial_transformation_matrixes=False))

    def detect(self, rgb, do_face=True):
        mp = self.mp
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        pose = None
        t0 = time.perf_counter()
        pr = self.pose.detect(img)
        self.timers["pose"] += time.perf_counter() - t0; self.timers["pose_n"] += 1
        if pr.pose_landmarks:
            pose = np.array([[p.x, p.y, p.z] for p in pr.pose_landmarks[0][:POSE_N]],
                            dtype=np.float32)
        hands = []
        t0 = time.perf_counter()
        hr = self.hand.detect(img)
        self.timers["hand"] += time.perf_counter() - t0; self.timers["hand_n"] += 1
        if hr.hand_landmarks:
            for k, lm in enumerate(hr.hand_landmarks[:2]):
                code = 0
                if hr.handedness and k < len(hr.handedness) and hr.handedness[k]:
                    code = 1 if hr.handedness[k][0].category_name.lower() == "left" else 2
                hands.append((np.array([[p.x, p.y, p.z] for p in lm[:HAND_N]],
                                       dtype=np.float32), code))
        face = None
        if do_face:
            t0 = time.perf_counter()
            fr = self.face.detect(img)
            self.timers["face"] += time.perf_counter() - t0; self.timers["face_n"] += 1
            if fr.face_landmarks:
                face = np.array([[p.x, p.y, p.z] for p in fr.face_landmarks[0][:FACE_N]],
                                dtype=np.float32)
        return pose, hands, face

    def close(self):
        for d in (self.pose, self.hand, self.face):
            try:
                d.close()
            except Exception:
                pass


class StubDetectors:
    """Deterministic fake detector for dry-runs/tests (no MediaPipe needed).
    Produces a plausible moving-hand signal derived from the frame content."""

    def __init__(self):
        self.calls = 0

    def detect(self, rgb, do_face=True):
        self.calls += 1
        m = float(rgb.mean()) / 255.0
        base = np.linspace(0.0, 1.0, POSE_N, dtype=np.float32)
        pose = np.stack([0.3 + 0.4 * base, 0.2 + 0.5 * base, 0 * base], axis=1)
        hands = []
        for s in range(2):
            ang = np.linspace(0, 2 * np.pi, HAND_N, dtype=np.float32)
            hl = np.stack([0.3 + 0.2 * s + 0.05 * np.cos(ang) + 0.1 * m,
                           0.5 + 0.05 * np.sin(ang), 0 * ang], axis=1).astype(np.float32)
            hands.append((hl, 1 + s))
        face = None
        if do_face:
            u = np.linspace(0, 1, FACE_N, dtype=np.float32)
            face = np.stack([0.45 + 0.1 * u + 0.02 * m, 0.2 + 0.2 * u, 0 * u], axis=1)
        return pose.astype(np.float32), hands, face


# --------------------------------------------------------------------------
# one clip -> raw landmark arrays
# --------------------------------------------------------------------------
def face_frame_indices(n_frames, k):
    return list(range(0, n_frames, max(1, int(k))))


def landmarks_from_frames(frames, detectors, face_every_k):
    n = len(frames)
    face_idx = face_frame_indices(n, face_every_k)
    tf = len(face_idx)
    pose = np.zeros((n, POSE_N, 3), np.float32)
    pose_ok = np.zeros(n, bool)
    hands = np.zeros((n, 2, HAND_N, 3), np.float32)
    hand_lab = np.zeros((n, 2), np.int8)
    face = np.zeros((tf, FACE_N, 3), np.float16)
    face_ok = np.zeros(tf, bool)
    fpos = {fi: j for j, fi in enumerate(face_idx)}
    for t, rgb in enumerate(frames):
        do_face = t in fpos
        p, hs, f = detectors.detect(rgb, do_face)
        if p is not None:
            pose[t] = p
            pose_ok[t] = True
        for slot, (hl, code) in enumerate(hs[:2]):   # detection order, NOT label
            hands[t, slot] = hl
            hand_lab[t, slot] = code
        if do_face and f is not None:
            face[fpos[t]] = f.astype(np.float16)
            face_ok[fpos[t]] = True
    return dict(pose=pose, pose_ok=pose_ok, hands=hands, hand_lab=hand_lab,
                face=face, face_ok=face_ok)


def empty_clip(n_frames, face_every_k):
    tf = len(face_frame_indices(n_frames, face_every_k))
    return dict(pose=np.zeros((n_frames, POSE_N, 3), np.float32),
                pose_ok=np.zeros(n_frames, bool),
                hands=np.zeros((n_frames, 2, HAND_N, 3), np.float32),
                hand_lab=np.zeros((n_frames, 2), np.int8),
                face=np.zeros((tf, FACE_N, 3), np.float16),
                face_ok=np.zeros(tf, bool))


def process_video_file(video_path, cfg, detectors):
    frames = sample_frames(video_path, cfg["n_frames"], cfg["max_side"])
    return landmarks_from_frames(frames, detectors, cfg["face_every_k"])


# --------------------------------------------------------------------------
# chunk processing (runs inside worker processes)
# --------------------------------------------------------------------------
_WORKER = {}


def init_worker(cfg, source, key):
    register_secret(key)
    _WORKER["cfg"] = cfg
    _WORKER["url"] = source
    _WORKER["zf"] = None
    _WORKER["det"] = None


def _get_zip():
    if _WORKER.get("zf") is None:
        _WORKER["zf"] = open_source(_WORKER["url"])
    return _WORKER["zf"]


def _get_det():
    if _WORKER.get("det") is None:
        c = _WORKER["cfg"]
        if c.get("detector") == "stub":
            _WORKER["det"] = StubDetectors()
        else:
            _WORKER["det"] = Detectors(c["model_dir"], c["pose_model"])
    return _WORKER["det"]


def _reset_connection():
    """Drop the (possibly stale) remote zip handle so the next chunk reconnects."""
    _WORKER["zf"] = None


def process_chunk(job):
    """job: dict(chunk_id, rows=[(path, ...)], out_path, manifest_sha1).
    Returns a small status dict. Raises nothing for content errors (recorded per
    clip); network errors abort the chunk (returned as status 'net_fail') so it
    is retried instead of being saved with holes."""
    cfg = _WORKER["cfg"]
    t0 = time.time()
    try:
        zf = _get_zip()
        det = _get_det()
        keys, errs, clips = [], [], []
        tmp_dir = cfg.get("tmp_dir") or tempfile.gettempdir()
        for row in job["rows"]:
            path = row["path"]
            key = hashlib.sha1(path.encode("utf-8")).hexdigest()
            keys.append(key)
            tmp_path = None
            try:
                data = read_member(zf, path)              # one Range request (or local read)
                fd, tmp_path = tempfile.mkstemp(suffix=".mp4", dir=tmp_dir)
                with os.fdopen(fd, "wb") as fh:
                    fh.write(data)
                clips.append(process_video_file(tmp_path, cfg, det))
                errs.append("")
            except NetworkError:
                raise
            except Exception as e:                        # bad/corrupt video etc.
                clips.append(empty_clip(cfg["n_frames"], cfg["face_every_k"]))
                errs.append(f"{e.__class__.__name__}: {redact(e)}"[:200])
            finally:
                if tmp_path and os.path.exists(tmp_path):
                    os.unlink(tmp_path)
        stack = {k: np.stack([c[k] for c in clips]) for k in clips[0]}
        out_path = Path(job["out_path"])
        tmp_out = out_path.with_name(out_path.name + ".tmp")
        with open(tmp_out, "wb") as fh:
            np.savez_compressed(
                fh, keys=np.array(keys), errs=np.array(errs),
                chunk_id=np.int32(job["chunk_id"]),
                cfg_fp=np.array(cfg["fingerprint"]),
                manifest_sha1=np.array(job["manifest_sha1"]),
                mediapipe=np.array(cfg.get("mediapipe_version", "")), **stack)
        os.replace(tmp_out, out_path)
        return dict(chunk_id=job["chunk_id"], status="ok", n=len(keys),
                    n_err=sum(1 for e in errs if e), seconds=time.time() - t0)
    except AuthError as e:
        _reset_connection()
        return dict(chunk_id=job["chunk_id"], status="auth_fail", error=redact(e))
    except NetworkError as e:
        _reset_connection()
        return dict(chunk_id=job["chunk_id"], status="net_fail", error=redact(e))
    except Exception as e:                                # unexpected: never leak the key
        return dict(chunk_id=job["chunk_id"], status="crash",
                    error=f"{e.__class__.__name__}: {redact(e)}"[:300])


# --------------------------------------------------------------------------
# benchmark + parity check (run BEFORE the long extraction)
# --------------------------------------------------------------------------
def benchmark(zf, paths, cfg, det, total_clips, workers, ks=(1, 2, 4, 8), log=print):
    """Measure decode / pose / hands / face cost on a few real clips, project the total
    time for several FACE_EVERY_K values, and measure how much the sparse-face
    interpolation changes the final features (same detections, so it isolates that choice)."""
    import sp_features
    tmp_dir = cfg.get("tmp_dir") or tempfile.gettempdir()
    n = cfg["n_frames"]
    t_dec, nclips, diffs, scale = 0.0, 0, [], []
    for p in paths:
        data = read_member(zf, p)
        fd, tp = tempfile.mkstemp(suffix=".mp4", dir=tmp_dir)
        with os.fdopen(fd, "wb") as fh:
            fh.write(data)
        try:
            t0 = time.perf_counter()
            frames = sample_frames(tp, n, cfg["max_side"])
            t_dec += time.perf_counter() - t0
            clip1 = landmarks_from_frames(frames, det, 1)        # face on EVERY frame
        finally:
            os.unlink(tp)
        nclips += 1
        k = cfg["face_every_k"]
        clipk = dict(clip1)
        clipk["face"], clipk["face_ok"] = clip1["face"][::k], clip1["face_ok"][::k]
        fa = sp_features.features_for_clip(clip1, 1)
        fb = sp_features.features_for_clip(clipk, k)
        diffs.append(float(np.abs(fa - fb).mean()))
        scale.append(float(np.abs(fa).mean()) + 1e-9)
    dec = t_dec / nclips
    tm = getattr(det, "timers", None)
    if not tm or not tm["pose_n"]:
        log("(stub detector: no per-model timings)")
        return None
    pose_ms = 1000 * tm["pose"] / tm["pose_n"]
    hand_ms = 1000 * tm["hand"] / tm["hand_n"]
    face_ms = 1000 * tm["face"] / max(tm["face_n"], 1)
    log(f"per clip: decode+resize {dec:.2f}s | per frame: pose {pose_ms:.0f} ms, hands {hand_ms:.0f} ms, face {face_ms:.0f} ms")
    log("projected single-process time per clip and total, if workers scale linearly:")
    best = None
    for k in ks:
        per = dec + n * (pose_ms + hand_ms) / 1000 + (-(-n // k)) * face_ms / 1000
        hrs = per * total_clips / max(workers, 1) / 3600
        mark = "  <- current" if k == cfg["face_every_k"] else ""
        log(f"  FACE_EVERY_K={k}: {per:5.2f} s/clip -> {hrs:6.1f} h total with {workers} worker(s){mark}")
    rel = float(np.mean(diffs) / np.mean(scale))
    log(f"parity: feature change from face every {cfg['face_every_k']} frames vs every frame = "
        f"{100*rel:.2f}% of mean feature magnitude (lower is better; <2% is negligible)")
    return dict(dec=dec, pose_ms=pose_ms, hand_ms=hand_ms, face_ms=face_ms, parity_rel=rel)


# --------------------------------------------------------------------------
# key handling + access pre-flight (never prints the key)
# --------------------------------------------------------------------------
def sanitize_key(raw):
    """Accept the bare key, a quoted key, 'key=...' or a whole pasted URL."""
    k = (raw or "").strip().strip("'\"").strip()
    m = re.search(r"[?&]key=([^&\s]+)", k)
    if m:
        k = m.group(1)
    if k.lower().startswith("key="):
        k = k[4:]
    return k.strip().strip("'\"")


def _probe(fn):
    try:
        r = fn()
        try:
            return r.status_code, dict(r.headers), r
        finally:
            r.close()
    except Exception as e:
        return f"{e.__class__.__name__}", {}, None


def preflight(url_template, key, log=print, timeout=30):
    """Test the portal the way the old notebook did (HEAD) AND the way the new one needs
    (ranged GET), then decide between 'stream' and 'archive' mode. Returns a dict with
    mode ('stream' | 'archive' | None), size and a human-readable reason."""
    import requests
    key = sanitize_key(key)
    register_secret(key)
    res = dict(mode=None, size=None, reason="")
    if not key:
        res["reason"] = ("NO KEY. Add a Colab secret named FDMSE_ISL_KEY (sidebar key icon), switch "
                         "'Notebook access' ON for this notebook, then rerun cell 2.")
        log(res["reason"])
        return res
    log(f"key: {len(key)} characters | letters/digits only: {key.isalnum()} | (contents never shown)")
    if not key.isalnum():
        log("  WARNING: the key contains characters other than letters/digits - it was probably "
            "copied with extra text. Re-copy ONLY the key itself.")
    url = url_template.format(key=key)

    h_status, h_hdr, _ = _probe(lambda: requests.head(url, timeout=timeout, allow_redirects=True))
    size = None
    try:
        size = int(h_hdr.get("Content-Length")) if h_hdr.get("Content-Length") else None
    except ValueError:
        pass
    log(f"HEAD            : {h_status}" + (f" | {size/2**30:.2f} GB | Accept-Ranges: {h_hdr.get('Accept-Ranges','-')}" if size else ""))

    g_status, g_hdr, _ = _probe(lambda: requests.get(url, headers={"Range": "bytes=0-0"},
                                                     stream=True, timeout=timeout))
    cr = g_hdr.get("Content-Range", "")
    log(f"GET with Range  : {g_status}" + (f" | Content-Range {cr}" if cr else ""))
    if g_status == 206 and "/" in cr and cr.split("/")[-1].isdigit():
        res.update(mode="stream", size=int(cr.split("/")[-1]),
                   reason="range requests work -> videos can be streamed")
        log("RESULT: OK - stream mode (no big download needed).")
        return res

    p_status, _, _ = _probe(lambda: requests.get(url, stream=True, timeout=timeout))
    log(f"GET plain       : {p_status}")
    if p_status == 200:
        res.update(mode="archive", size=size,
                   reason="portal allows normal downloads but not range requests -> download the archive once")
        log("RESULT: OK - archive mode. Run the 'DOWNLOAD ARCHIVE' cell once (resumable).")
        return res

    codes = {h_status, g_status, p_status}
    if codes <= {401, 403}:
        res["reason"] = (
            "KEY REJECTED: the portal answers 403 to every request type.\n"
            "  1) Colab sidebar -> key icon: the secret must be named exactly FDMSE_ISL_KEY and the "
            "'Notebook access' switch must be ON for THIS notebook (it is per notebook).\n"
            "  2) The value must be only the key. Compare its length above with the key you were given.\n"
            "  3) Open the same download link in your own browser. If it refuses there too, the key "
            "expired or hit a limit: email the dataset authors and ask them to renew it.")
    elif isinstance(h_status, int) and h_status < 400:
        res["reason"] = ("The portal accepts the key for information requests (HEAD) but REFUSES downloads. "
                         "This usually means the key has a download limit that is used up. Ask the authors "
                         "to reset or renew it.")
    elif any(isinstance(c, str) for c in codes):
        res["reason"] = ("Could not reach the portal (network error). Check that Colab can reach "
                         "cs.rkmvu.ac.in, then rerun this cell.")
    else:
        res["reason"] = f"Unexpected portal answers {sorted(map(str, codes))}. Send this output to your helper."
    log("RESULT: FAILED\n" + res["reason"])
    return res


def check_archive(path, min_videos=1, log=print):
    """Validate a locally downloaded FDMSE-ISL zip and give a clear message if it is incomplete."""
    p = Path(path)
    if not p.exists():
        raise SystemExit(f"Archive not found: {p}")
    size = p.stat().st_size
    try:
        with zipfile.ZipFile(p) as zf:
            names = zf.namelist()
    except zipfile.BadZipFile:
        raise SystemExit(
            f"{p.name} ({size/2**30:.2f} GB) is not a complete zip: the index at the END of the file is "
            "missing, which is what a partly downloaded or partly uploaded file looks like. Re-download "
            "or re-upload it fully, then rerun.")
    n_mp4 = sum(1 for n in names if n.lower().endswith(".mp4"))
    metas = [n for n in names if os.path.basename(n).lower() == "metadata.csv"]
    log(f"archive: {size/2**30:.2f} GB | {n_mp4} videos | metadata.csv files: {len(metas)}")
    if n_mp4 < min_videos or not metas:
        raise SystemExit("This zip does not look like the FDMSE-ISL archive (no videos or no metadata.csv).")
    if n_mp4 < 40000:
        log(f"  NOTE: the full dataset has about 40,033 videos; this archive has {n_mp4}.")
    return n_mp4


In [ ]:
%%writefile sp_features.py
"""SignSage features: sign_features_v3 (VERBATIM from the original notebook /
backend/app/ml/preprocessing/sign_features_v3.py) + glue that rebuilds the
543-landmark layout from the raw landmarks saved by sp_core.py."""
import numpy as np

N_FRAMES = 60
FEAT_DIM = 122
MAX_GAP = 5
EPS = 1e-6

_LHAND = slice(501, 522)   # MediaPipe slot 0 (label ignored)
_RHAND = slice(522, 543)   # MediaPipe slot 1 (label ignored)
_FACE_NOSE = 1             # face mesh index; face array is landmarks[:,33:500,:2]
_FACE_CHIN = 152
_FACE_MOUTH = 13
_FACE_FOREHEAD = 10
_L_SHOULDER = 11
_R_SHOULDER = 12
_WRIST = 0
_INDEX_TIP = 8
_MIDDLE_TIP = 12
_FACE_POINTS = (_FACE_NOSE, _FACE_CHIN, _FACE_MOUTH, _FACE_FOREHEAD)


def _xy(landmarks, sl):
    return landmarks[:, sl, :2]


def _present(hand_xy):
    return np.any(hand_xy != 0.0, axis=(1, 2))


def _fill_gaps(hand_xy, max_gap=MAX_GAP):
    T = hand_xy.shape[0]
    filled = hand_xy.copy().astype(np.float64)
    present = _present(hand_xy)
    mask = present.copy().astype(np.float64)
    if T == 0:
        return filled, mask
    miss = ~present
    run_start = None
    for t in range(T + 1):
        if t < T and miss[t] and run_start is None:
            run_start = t
        if (t == T or not miss[t]) and run_start is not None:
            s, e = run_start, t - 1
            length = e - s + 1
            if length <= max_gap and s > 0 and e < T - 1:
                a = filled[s - 1]; b = filled[e + 1]
                frac = ((np.arange(s, e + 1, dtype=np.float64) - (s - 1))
                        / ((e + 1) - (s - 1)))
                for k in range(s, e + 1):
                    fk = frac[k - s]
                    filled[k] = a + fk * (b - a)
                mask[s:e + 1] = 1.0
            run_start = None
    return filled, mask


def _resample(feat, n_frames):
    T = feat.shape[0]
    if T == 0:
        return np.zeros((n_frames, FEAT_DIM), dtype=np.float32)
    if T == n_frames:
        return feat
    if T > n_frames:
        idx = np.linspace(0, T - 1, n_frames).round().astype(int)
        return feat[idx]
    pad = np.repeat(feat[-1:], n_frames - T, axis=0)
    return np.concatenate([feat, pad], axis=0)


def _shoulder_centre_width(pose):
    lsh = pose[:, _L_SHOULDER, :2]
    rsh = pose[:, _R_SHOULDER, :2]
    centre = (lsh + rsh) / 2.0
    width = np.linalg.norm(lsh - rsh, axis=1, keepdims=True) + EPS
    return centre, width


def _mirror_x(arr, mid_x):
    out = arr.copy()
    out[:, :, 0] = 2.0 * mid_x[:, None] - out[:, :, 0]
    return out


def _track_hands(lh_xy, rh_xy):
    T = lh_xy.shape[0]
    lh_present = _present(lh_xy)
    rh_present = _present(rh_xy)
    phys = []
    slots = [None, None]
    for t in range(T):
        detected = []
        if lh_present[t]:
            detected.append((lh_xy[t, _WRIST].copy(), lh_xy[t].copy()))
        if rh_present[t]:
            detected.append((rh_xy[t, _WRIST].copy(), rh_xy[t].copy()))
        phys_t = [None, None]
        if len(detected) == 0:
            pass
        elif len(detected) == 1:
            wrist, lm = detected[0]
            best_s, best_d = None, None
            for s in (0, 1):
                if slots[s] is not None:
                    d = float(np.linalg.norm(wrist - slots[s]["wrist"]))
                    if best_d is None or d < best_d:
                        best_s, best_d = s, d
            if best_s is None:
                best_s = 0
            phys_t[best_s] = lm
            slots[best_s] = {"wrist": wrist, "lm": lm}
        else:
            d0, d1 = detected
            if d0[0][0] > d1[0][0]:
                d0, d1 = d1, d0
            if slots[0] is not None and slots[1] is not None:
                c1 = (np.linalg.norm(d0[0] - slots[0]["wrist"])
                      + np.linalg.norm(d1[0] - slots[1]["wrist"]))
                c2 = (np.linalg.norm(d0[0] - slots[1]["wrist"])
                      + np.linalg.norm(d1[0] - slots[0]["wrist"]))
                if c1 <= c2:
                    phys_t[0], phys_t[1] = d0[1], d1[1]
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[0], phys_t[1] = d1[1], d0[1]
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
            elif slots[0] is not None:
                if np.linalg.norm(d0[0] - slots[0]["wrist"]) <= np.linalg.norm(d1[0] - slots[0]["wrist"]):
                    phys_t[0], phys_t[1] = d0[1], d1[1]
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[0], phys_t[1] = d1[1], d0[1]
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
            elif slots[1] is not None:
                if np.linalg.norm(d0[0] - slots[1]["wrist"]) <= np.linalg.norm(d1[0] - slots[1]["wrist"]):
                    phys_t[1], phys_t[0] = d0[1], d1[1]
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[1], phys_t[0] = d1[1], d0[1]
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
            else:
                phys_t[0], phys_t[1] = d0[1], d1[1]
                slots[0] = {"wrist": d0[0], "lm": d0[1]}
                slots[1] = {"wrist": d1[0], "lm": d1[1]}
        phys.append(phys_t)
    wrists = np.zeros((T, 2, 2), dtype=np.float64)
    for t, phys_t in enumerate(phys):
        for s in (0, 1):
            if phys_t[s] is not None:
                wrists[t, s] = phys_t[s][_WRIST]
    return phys, wrists


def _active_secondary(phys, wrists):
    T = wrists.shape[0]
    travel = np.zeros(2)
    n_present = np.zeros(2, dtype=int)
    for s in (0, 1):
        present = np.any(wrists[:, s] != 0.0, axis=1)
        n_present[s] = int(present.sum())
        if n_present[s] > 1:
            travel[s] = float(np.sum(np.linalg.norm(np.diff(wrists[:, s], axis=0), axis=1)))
    if travel[0] > travel[1]:
        return 0
    if travel[1] > travel[0]:
        return 1
    return 1 if n_present[1] > n_present[0] else 0


def _should_mirror(phys, active, wrists, centre):
    T = wrists.shape[0]
    active_present = np.any(wrists[:, active] != 0.0, axis=1)
    sh_valid = np.any(centre != 0.0, axis=1)
    valid = active_present & sh_valid
    if valid.sum() < 2:
        return False
    return float(np.mean(wrists[valid, active, 0])) < float(np.mean(centre[valid, 0]))


def extract_features(landmarks, layout='signsage', n_frames=N_FRAMES):
    if layout != 'signsage':
        raise ValueError("sign_features_v3 supports layout='signsage' only (543-array)")
    if landmarks.ndim != 3 or landmarks.shape[2] < 2:
        raise ValueError("landmarks must be (T, N, >=3)")
    T = landmarks.shape[0]
    pose = landmarks[:, 0:33, :]
    lh = _xy(landmarks, _LHAND)
    rh = _xy(landmarks, _RHAND)
    face = landmarks[:, 33:500, :2]

    centre, width = _shoulder_centre_width(pose)
    phys, wrists = _track_hands(lh, rh)
    active = _active_secondary(phys, wrists)
    secondary = 1 - active

    mirror = _should_mirror(phys, active, wrists, centre)
    if mirror:
        mid_x = centre[:, 0]
        lh = _mirror_x(lh, mid_x)
        rh = _mirror_x(rh, mid_x)
        face = _mirror_x(face, mid_x)
        phys, wrists = _track_hands(lh, rh)
        active = _active_secondary(phys, wrists)
        secondary = 1 - active

    def hand_part(hfilled, hmask, hsize):
        wrist = hfilled[:, _WRIST, :]
        others = np.delete(hfilled, _WRIST, axis=1)
        wrs = (wrist - centre) / width
        orw = ((others - wrist[:, None, :]) / hsize.reshape(-1, 1, 1))
        wrs = wrs * hmask[:, None]
        orw = orw * hmask[:, None, None]
        return np.concatenate([wrs.reshape(T, -1), orw.reshape(T, -1)], axis=1)

    slot_lm = [np.zeros((T, 21, 2), dtype=np.float64),
               np.zeros((T, 21, 2), dtype=np.float64)]
    for t, phys_t in enumerate(phys):
        for s in (0, 1):
            if phys_t[s] is not None:
                slot_lm[s][t] = phys_t[s]

    l_filled, l_mask = _fill_gaps(slot_lm[0])
    r_filled, r_mask = _fill_gaps(slot_lm[1])

    def hand_size(hfilled):
        return np.linalg.norm(hfilled[:, _WRIST, :] - hfilled[:, _MIDDLE_TIP, :],
                              axis=1, keepdims=True) + EPS

    active_filled = l_filled if active == 0 else r_filled
    active_mask = l_mask if active == 0 else r_mask
    active_size = hand_size(active_filled)
    sec_filled = l_filled if secondary == 0 else r_filled
    sec_mask = l_mask if secondary == 0 else r_mask
    sec_size = hand_size(sec_filled)

    active_hand = hand_part(active_filled, active_mask, active_size)
    sec_hand = hand_part(sec_filled, sec_mask, sec_size)

    h2f = []
    for hp, hmask in ((active_filled[:, _WRIST, :], active_mask),
                      (active_filled[:, _INDEX_TIP, :], active_mask),
                      (sec_filled[:, _WRIST, :], sec_mask),
                      (sec_filled[:, _INDEX_TIP, :], sec_mask)):
        for fidx in _FACE_POINTS:
            h2f.append(((hp - face[:, fidx, :]) / width) * hmask[:, None])
    h2f = np.concatenate(h2f, axis=1)

    def wrist_vel(hfilled, hmask):
        wrist = hfilled[:, _WRIST, :]
        vel = np.zeros_like(wrist)
        if T > 1:
            vel[1:] = wrist[1:] - wrist[:-1]
        return vel * hmask[:, None]

    active_vel = wrist_vel(active_filled, active_mask)
    sec_vel = wrist_vel(sec_filled, sec_mask)

    feat = np.concatenate([active_hand, sec_hand,
                           active_mask.reshape(T, 1), sec_mask.reshape(T, 1),
                           h2f,
                           active_vel, sec_vel], axis=1)
    assert feat.shape[1] == FEAT_DIM, feat.shape[1]
    feat = _resample(feat, n_frames)
    return feat.astype(np.float32)


# ==========================================================================
# Glue (new): raw saved landmarks -> (T, 543, 3) layout -> v3 features
# ==========================================================================
def _nearest_valid(ok):
    valid = np.flatnonzero(ok)
    if valid.size == 0:
        return None
    t = np.arange(ok.shape[0])
    return valid[np.abs(t[:, None] - valid[None, :]).argmin(axis=1)]


def interpolate_face(face, face_ok, n_frames, face_every_k):
    """Linear interpolation of the (sparse) face detections to every frame."""
    fidx = np.arange(0, n_frames, max(1, int(face_every_k)))
    good = np.flatnonzero(face_ok)
    if good.size == 0:
        return np.zeros((n_frames, 468, 3), np.float32)
    fx = fidx[good].astype(np.float64)
    F = face[good].astype(np.float32)
    t = np.arange(n_frames, dtype=np.float64)
    if good.size == 1:
        return np.repeat(F[:1], n_frames, axis=0)
    right = np.clip(np.searchsorted(fx, t, side="left"), 1, len(fx) - 1)
    left = right - 1
    w = np.clip((t - fx[left]) / (fx[right] - fx[left]), 0.0, 1.0).astype(np.float32)
    return (1.0 - w)[:, None, None] * F[left] + w[:, None, None] * F[right]


def to_landmarks_543(clip, face_every_k):
    """clip: dict with pose,pose_ok,hands,face,face_ok for ONE video."""
    n = clip["pose"].shape[0]
    out = np.zeros((n, 543, 3), np.float32)
    pose = clip["pose"].copy()
    near = _nearest_valid(clip["pose_ok"])
    if near is not None:
        pose = pose[near]                      # fill missed pose frames from neighbours
    out[:, 0:33] = pose
    out[:, 33:501] = interpolate_face(clip["face"], clip["face_ok"], n, face_every_k)
    out[:, 501:522] = clip["hands"][:, 0]
    out[:, 522:543] = clip["hands"][:, 1]
    return out


def clip_quality(clip):
    hands_any = np.any(clip["hands"] != 0.0, axis=(2, 3)).any(axis=1)
    return dict(pose_ok_frac=float(clip["pose_ok"].mean()),
                hand_frac=float(hands_any.mean()),
                face_ok_frac=float(clip["face_ok"].mean()))


def is_usable(q, min_pose=0.5, min_hand=0.25):
    return q["pose_ok_frac"] >= min_pose and q["hand_frac"] >= min_hand


def features_for_clip(clip, face_every_k):
    return extract_features(to_landmarks_543(clip, face_every_k))


In [ ]:
%%writefile sp_run.py
"""SignSage FDMSE-ISL runner (v2): manifest, resumable chunk scheduling, status,
verification and feature building. Pure orchestration - heavy lifting is in
sp_core.py (extraction) and sp_features.py (features)."""
from __future__ import annotations

import concurrent.futures as cf
import hashlib
import io
import json
import multiprocessing as mp
import os
import re
import shutil
import time
from concurrent.futures.process import BrokenProcessPool
from pathlib import Path

import numpy as np
import pandas as pd

import sp_core
import sp_features

CHUNK_RE = re.compile(r"^chunk_(\d{5})\.npz$")
FEAT_RE = re.compile(r"^feat_chunk_(\d{5})\.npz$")
EXPECTED = {"train": 20016, "validation": 4003, "test": 16014}


# --------------------------------------------------------------------------
# config
# --------------------------------------------------------------------------
def make_cfg(n_frames, face_every_k, pose_model, max_side, model_dir, tmp_dir,
             detector="mediapipe", mediapipe_version=""):
    core = dict(n_frames=int(n_frames), face_every_k=int(face_every_k),
                pose_model=str(pose_model), max_side=int(max_side or 0),
                feature_spec="sign_features_v3", detector=detector)
    fp = hashlib.sha1(json.dumps(core, sort_keys=True).encode()).hexdigest()[:12]
    return dict(core, fingerprint=fp, model_dir=str(model_dir), tmp_dir=str(tmp_dir),
                mediapipe_version=mediapipe_version)


def check_or_write_config(meta_dir, cfg):
    """Refuse to mix chunks produced with different extraction settings."""
    meta_dir = Path(meta_dir)
    meta_dir.mkdir(parents=True, exist_ok=True)
    p = meta_dir / "extraction_config.json"
    public = {k: v for k, v in cfg.items() if k not in ("model_dir", "tmp_dir")}
    if p.exists():
        old = json.loads(p.read_text())
        if old.get("fingerprint") != cfg["fingerprint"]:
            raise SystemExit(
                "EXTRACTION SETTINGS CHANGED.\n"
                f"  saved   : {json.dumps({k: old.get(k) for k in ('n_frames','face_every_k','pose_model','max_side')})}\n"
                f"  current : {json.dumps({k: cfg.get(k) for k in ('n_frames','face_every_k','pose_model','max_side')})}\n"
                "Mixing settings would silently corrupt the dataset. Restore the saved values in the "
                "config cell, or point OUT_ROOT to a NEW folder to start over.")
        if old.get("mediapipe_version") and cfg.get("mediapipe_version") \
                and old["mediapipe_version"] != cfg["mediapipe_version"]:
            print(f"WARNING: saved with mediapipe {old['mediapipe_version']} but this machine has "
                  f"{cfg['mediapipe_version']}. Prefer: pip install mediapipe=={old['mediapipe_version']}")
    else:
        p.write_text(json.dumps(public, indent=2))
    return public


# --------------------------------------------------------------------------
# manifest (deterministic: identical on every machine)
# --------------------------------------------------------------------------
def _norm_split(v):
    v = str(v).strip().lower()
    if "val" in v or "dev" in v:
        return "validation"
    if "test" in v:
        return "test"
    if "train" in v:
        return "train"
    return v


def build_manifest(zf, chunk_size=50, seed=42, log=print):
    names = [i.filename for i in zf.infolist()]
    mp4 = {n for n in names if n.lower().endswith(".mp4")}
    by_base = {}
    for n in mp4:
        by_base.setdefault(os.path.basename(n), []).append(n)
    metas = [n for n in names if os.path.basename(n).lower() == "metadata.csv"]
    if len(metas) > 1:                                   # prefer the full-dataset file over a 400-class subset
        metas = [n for n in metas if "400" not in n] or metas
    meta_name = metas[0] if metas else None
    if meta_name is None:
        cands = [n for n in names if n.lower().endswith((".csv", ".txt"))][:20]
        raise SystemExit(f"metadata.csv not found in the archive. Candidates: {cands}")
    df = pd.read_csv(io.BytesIO(zf.read(meta_name)))
    df.columns = [c.strip().lower() for c in df.columns]
    log(f"metadata: {meta_name} | columns {list(df.columns)} | rows {len(df)}")
    for need in ("class", "split"):
        if need not in df.columns:
            raise SystemExit(f"metadata.csv has no '{need}' column: {list(df.columns)}")

    def resolve(row):
        vd = str(row["video_dir"]) if "video_dir" in df.columns else ""
        vn = str(row["video_name"]) if "video_name" in df.columns else ""
        cands = [vd, "FDMSE-ISL/" + vd.lstrip("/"),
                 vd.rstrip("/") + "/" + vn, "FDMSE-ISL/" + vd.strip("/") + "/" + vn]
        for c in cands:
            if c in mp4:
                return c
        for b in (vn, os.path.basename(vd)):
            if b in by_base and len(by_base[b]) == 1:
                return by_base[b][0]
        return None

    df["path"] = df.apply(resolve, axis=1)
    cov = df["path"].notna().mean()
    log(f"path coverage: {cov:.2%} ({int(df['path'].notna().sum())}/{len(df)})")
    if cov < 0.999:
        raise SystemExit("Could not map metadata rows to archive videos. Sample rows:\n"
                         + df[df["path"].isna()].head(3).to_string())
    df = df.dropna(subset=["path"]).drop_duplicates("path").copy()
    df["class"] = df["class"].astype(str).str.strip()
    df["split"] = df["split"].map(_norm_split)
    base = df["path"].map(os.path.basename)
    df["signer"] = base.str.extract(r"(s\d+)_", expand=False).fillna("unknown")
    df["word_id"] = base.str.extract(r"_w(\d+)", expand=False).fillna("-1").astype(int)
    classes = sorted(df["class"].unique())
    df["class_id"] = df["class"].map({c: i for i, c in enumerate(classes)})

    # ---- structural checks (never silently skipped) ----
    sig = {s: set(df.loc[df.split == s, "signer"]) for s in ("train", "validation", "test")}
    assert not (sig["train"] & sig["validation"] or sig["train"] & sig["test"]
                or sig["validation"] & sig["test"]), "signers overlap across splits!"
    counts = df.split.value_counts().to_dict()
    log(f"videos: {len(df)} | classes: {len(classes)} | split counts: {counts}")
    log(f"signers per split: { {k: len(v) for k, v in sig.items()} } (published: 10/2/8)")
    for k, v in EXPECTED.items():
        if counts.get(k) != v:
            log(f"  NOTE: {k} has {counts.get(k)} videos, published {v}")

    # ---- order: every class appears early, so a partial run is still balanced ----
    df = df.sort_values("path").reset_index(drop=True)
    perm = np.random.RandomState(seed).permutation(len(df))
    df = df.iloc[perm].reset_index(drop=True)
    df["rank_in_class"] = df.groupby("class").cumcount()
    df = df.sort_values(["rank_in_class", "class_id"], kind="mergesort").reset_index(drop=True)
    df["idx"] = np.arange(len(df))
    df["chunk_id"] = df["idx"] // int(chunk_size)
    keep = ["idx", "chunk_id", "path", "class", "class_id", "word_id", "signer", "split"]
    return df[keep], classes


def manifest_sha1(df):
    return hashlib.sha1(df.to_csv(index=False).encode("utf-8")).hexdigest()


def load_or_build_manifest(meta_dir, source, chunk_size, seed=42, log=print):
    meta_dir = Path(meta_dir)
    meta_dir.mkdir(parents=True, exist_ok=True)
    mpath = meta_dir / "manifest.csv"
    if mpath.exists():
        df = pd.read_csv(mpath)
        if int(df["chunk_id"].max() + 1) != -(-len(df) // int(chunk_size)):
            raise SystemExit("CHUNK_SIZE differs from the saved manifest. Restore the old value "
                             "(or use a new OUT_ROOT).")
        log(f"manifest loaded: {len(df)} videos, {df['chunk_id'].nunique()} chunks")
    else:
        log("building manifest from the archive index (remote: reads ~5 MB only)...")
        zf = sp_core.open_source(source)
        df, classes = build_manifest(zf, chunk_size, seed, log)
        zf.close()
        df.to_csv(mpath, index=False)
        (meta_dir / "label_map.json").write_text(
            json.dumps({"classes": classes}, ensure_ascii=False, indent=1), encoding="utf-8")
        log(f"manifest saved: {mpath}")
    sha = manifest_sha1(df)
    (meta_dir / "manifest.sha1").write_text(sha)
    log(f"manifest sha1: {sha[:16]}...  (must match on every machine)")
    return df, sha


# --------------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------------
def list_ids(directory, pattern=CHUNK_RE):
    d = Path(directory)
    if not d.is_dir():
        return set()
    out = set()
    for p in d.iterdir():
        m = pattern.match(p.name)
        if m:
            out.add(int(m.group(1)))
    return out


def chunk_path(directory, cid):
    return Path(directory) / f"chunk_{cid:05d}.npz"


def atomic_copy(src, dst, retries=5):
    dst = Path(dst)
    tmp = dst.with_name(dst.name + ".tmp")
    last = None
    for a in range(retries):
        try:
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
            return
        except Exception as e:                 # Drive FUSE hiccups
            last = e
            time.sleep(2 * (a + 1))
    raise RuntimeError(f"could not copy {src} -> {dst}: {last}")


def clean_stale_tmp(directory):
    d = Path(directory)
    if d.is_dir():
        for p in d.glob("*.tmp"):
            try:
                p.unlink()
            except OSError:
                pass


def adopt_local_chunks(local_dir, out_dir, log=print):
    """A worker may finish its chunk just after you stop the run. Such a chunk sits on the
    local disk but never reached Drive; copy it over instead of redoing the work."""
    local_dir, out_dir = Path(local_dir), Path(out_dir)
    if local_dir.resolve() == out_dir.resolve() or not local_dir.is_dir():
        return 0
    n = 0
    for cid in sorted(list_ids(local_dir)):
        src = chunk_path(local_dir, cid)
        if chunk_path(out_dir, cid).exists():
            src.unlink(missing_ok=True)
            continue
        try:
            z = np.load(src, allow_pickle=False)
            _ = z["pose"].shape, z["keys"].shape
            z.close()
        except Exception:
            src.unlink(missing_ok=True)
            continue
        atomic_copy(src, chunk_path(out_dir, cid))
        src.unlink(missing_ok=True)
        n += 1
    if n:
        log(f"recovered {n} finished chunk(s) left on the local disk from the previous session")
    return n


def _kill_workers(pool):
    for p in list(getattr(pool, "_processes", {}).values()):
        try:
            p.terminate()
        except Exception:
            pass


# --------------------------------------------------------------------------
# the resumable run loop
# --------------------------------------------------------------------------
def run_extraction(manifest, sha, cfg, source, key, out_chunks, local_chunks,
                   machine_index=0, machine_count=1, workers=2, max_hours=None,
                   limit_chunks=None, max_attempts=3, order="forward", log=print):
    url = source
    out_chunks, local_chunks = Path(out_chunks), Path(local_chunks)
    out_chunks.mkdir(parents=True, exist_ok=True)
    local_chunks.mkdir(parents=True, exist_ok=True)
    clean_stale_tmp(out_chunks)
    clean_stale_tmp(local_chunks)
    direct = out_chunks.resolve() == local_chunks.resolve()
    adopt_local_chunks(local_chunks, out_chunks, log)

    n_chunks = int(manifest["chunk_id"].max()) + 1
    done = list_ids(out_chunks)
    done = list_ids(out_chunks)
    mine = [c for c in range(n_chunks) if c % machine_count == machine_index and c not in done]
    others = [c for c in range(n_chunks) if c % machine_count != machine_index and c not in done]
    if order == "reverse":
        mine.reverse()
    if limit_chunks:
        mine = mine[:int(limit_chunks)]
    if machine_count > 1 and others:
        log(f"NOTE: MACHINE_COUNT={machine_count} -> this machine (index {machine_index}) only does its own "
            f"share. {len(others)} unfinished chunk(s) belong to the other machine(s). To do everything "
            f"here instead, set MACHINE_COUNT = 1.")
    log(f"chunks total={n_chunks} | already done (any machine)={len(done)} | "
        f"to do on this machine={len(mine)} | workers={workers}")
    if not mine:
        log("nothing to do on this machine - all assigned chunks are finished.")
        return dict(done=0, failed=[], stopped="complete")

    rows_by_chunk = {c: g[["path"]].to_dict("records") for c, g in manifest.groupby("chunk_id")}
    queue = list(mine)
    attempts = {c: 0 for c in mine}
    failed, completed = [], 0
    t0 = time.time()
    deadline = t0 + max_hours * 3600 if max_hours else None
    stopped = "complete"
    restarts = 0
    ctx = mp.get_context("spawn")

    def make_pool():
        return cf.ProcessPoolExecutor(max_workers=workers, mp_context=ctx,
                                      initializer=sp_core.init_worker,
                                      initargs=(cfg, url, key))

    pool = make_pool()
    inflight = {}

    def submit_next():
        while queue and len(inflight) < workers:
            if deadline and time.time() > deadline:
                return False
            c = queue.pop(0)
            attempts[c] += 1
            job = dict(chunk_id=c, rows=rows_by_chunk[c], manifest_sha1=sha,
                       out_path=str(chunk_path(local_chunks, c)))
            inflight[pool.submit(sp_core.process_chunk, job)] = c
        return True

    try:
        submit_next()
        while inflight:
            finished, _ = cf.wait(list(inflight), timeout=30, return_when=cf.FIRST_COMPLETED)
            for fut in finished:
                c = inflight.pop(fut)
                try:
                    res = fut.result()
                except BrokenProcessPool:
                    restarts += 1
                    log(f"worker process died (out of memory?) - restart {restarts}/5")
                    if restarts > 5:
                        raise
                    queue.insert(0, c)
                    attempts[c] -= 1
                    for f2, c2 in list(inflight.items()):
                        queue.insert(0, c2)
                        attempts[c2] -= 1
                        inflight.pop(f2)
                    pool.shutdown(wait=False, cancel_futures=True)
                    pool = make_pool()
                    break
                st = res["status"]
                if st == "ok":
                    if not direct:
                        atomic_copy(chunk_path(local_chunks, c), chunk_path(out_chunks, c))
                        chunk_path(local_chunks, c).unlink(missing_ok=True)
                    completed += 1
                    el = time.time() - t0
                    per = el / completed
                    left = len(mine) - completed
                    gd = len(done) + completed
                    log(f"[{completed}/{len(mine)}] chunk {c:05d} ok "
                        f"({res['n_err']} unreadable clips, {res['seconds']:.0f}s) | "
                        f"all machines {gd}/{n_chunks} ({100*gd/n_chunks:.1f}%) | "
                        f"ETA this machine ~{per*left/3600:.1f} h")
                elif st == "auth_fail":
                    log(f"STOP: portal rejected the key ({res['error']}). Check the secret.")
                    stopped = "auth"
                    queue.clear()
                else:
                    log(f"chunk {c:05d} {st}: {res.get('error','')} (attempt {attempts[c]}/{max_attempts})")
                    if attempts[c] < max_attempts:
                        queue.append(c)
                        time.sleep(5)
                    else:
                        failed.append((c, st))
            if stopped == "auth":
                break
            if not submit_next() and not inflight:
                stopped = "time_limit"
                break
        if deadline and time.time() > deadline and queue:
            stopped = "time_limit"
    except KeyboardInterrupt:
        stopped = "interrupted"
        log("interrupted - finished chunks are safe; the chunk in progress will be redone next run.")
        _kill_workers(pool)
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    log(f"session finished: {completed} chunk(s) done, {len(failed)} failed, stopped={stopped}, "
        f"{(time.time()-t0)/60:.1f} min")
    return dict(done=completed, failed=failed, stopped=stopped)


# --------------------------------------------------------------------------
# status / verification
# --------------------------------------------------------------------------
def status(manifest, out_chunks, machine_count=None, log=print):
    done = list_ids(out_chunks)
    n_chunks = int(manifest["chunk_id"].max()) + 1
    m = manifest[manifest["chunk_id"].isin(done)]
    log(f"chunks finished: {len(done)}/{n_chunks} ({100*len(done)/n_chunks:.1f}%) | "
        f"videos: {len(m)}/{len(manifest)}")
    per_split = m.split.value_counts().reindex(["train", "validation", "test"]).fillna(0).astype(int)
    log(f"finished videos per split: {per_split.to_dict()}")
    cls = m.groupby("class_id").size()
    log(f"classes with >=1 finished video: {len(cls)}/{manifest['class_id'].nunique()} | "
        f"min/median/max per class: {int(cls.min()) if len(cls) else 0}/"
        f"{int(cls.median()) if len(cls) else 0}/{int(cls.max()) if len(cls) else 0}")
    missing = sorted(set(range(n_chunks)) - done)
    return dict(done=len(done), total=n_chunks, missing=missing)


def verify_chunks(out_chunks, sha, cfg_fp, delete_bad=True, log=print):
    bad, wrong = [], []
    for cid in sorted(list_ids(out_chunks)):
        p = chunk_path(out_chunks, cid)
        try:
            z = np.load(p, allow_pickle=False)
            _ = z["pose"].shape, z["hands"].shape, z["face"].shape, z["keys"].shape
            if str(z["manifest_sha1"]) != sha or str(z["cfg_fp"]) != cfg_fp:
                wrong.append(cid)
            z.close()
        except Exception as e:
            bad.append(cid)
            log(f"corrupt: {p.name} ({e.__class__.__name__})")
            if delete_bad:
                p.unlink(missing_ok=True)
    log(f"verified: corrupt={len(bad)} (deleted={delete_bad}) | from a different manifest/settings={len(wrong)}")
    if wrong:
        log(f"  WARNING chunks with mismatched manifest/settings: {wrong[:10]} ...")
    return bad, wrong


# --------------------------------------------------------------------------
# features from the saved landmarks (fast, CPU, resumable)
# --------------------------------------------------------------------------
def build_features(out_chunks, out_features, cfg, min_pose=0.5, min_hand=0.25, log=print):
    out_features = Path(out_features)
    out_features.mkdir(parents=True, exist_ok=True)
    clean_stale_tmp(out_features)
    have = list_ids(out_features, FEAT_RE)
    todo = sorted(list_ids(out_chunks) - have)
    log(f"feature files existing={len(have)} | chunks to convert={len(todo)}")
    t0 = time.time()
    k = cfg["face_every_k"]
    for n, cid in enumerate(todo, 1):
        z = np.load(chunk_path(out_chunks, cid), allow_pickle=False)
        keys, errs = z["keys"].astype(str), z["errs"].astype(str)
        cnt = len(keys)
        X = np.zeros((cnt, sp_features.N_FRAMES, sp_features.FEAT_DIM), np.float32)
        usable = np.zeros(cnt, bool)
        q = np.zeros((cnt, 3), np.float32)
        for i in range(cnt):
            if errs[i]:
                continue
            clip = {name: z[name][i] for name in ("pose", "pose_ok", "hands", "face", "face_ok")}
            qi = sp_features.clip_quality(clip)
            q[i] = (qi["pose_ok_frac"], qi["hand_frac"], qi["face_ok_frac"])
            if not sp_features.is_usable(qi, min_pose, min_hand):
                continue
            f = sp_features.features_for_clip(clip, k)
            if np.isfinite(f).all() and np.abs(f).sum() > 0:
                X[i] = f
                usable[i] = True
        out = chunk_path(out_features, cid).with_name(f"feat_chunk_{cid:05d}.npz")
        tmp = out.with_name(out.name + ".tmp")
        with open(tmp, "wb") as fh:
            np.savez_compressed(fh, keys=keys, X=X, usable=usable, quality=q, errs=errs,
                                cfg_fp=np.array(cfg["fingerprint"]))
        os.replace(tmp, out)
        z.close()
        if n % 25 == 0 or n == len(todo):
            log(f"  features: {n}/{len(todo)} chunks ({(time.time()-t0)/60:.1f} min)")
    total = list_ids(out_features, FEAT_RE)
    log(f"feature files now: {len(total)}")
    return len(todo)


def feature_summary(manifest, out_features, log=print):
    ids = sorted(list_ids(out_features, FEAT_RE))
    if not ids:
        log("no feature files yet")
        return None
    kk, uu = [], []
    for cid in ids:
        z = np.load(Path(out_features) / f"feat_chunk_{cid:05d}.npz", allow_pickle=False)
        kk += list(z["keys"].astype(str)); uu += list(z["usable"])
        z.close()
    key_of = {hashlib.sha1(p.encode()).hexdigest(): i for i, p in enumerate(manifest["path"])}
    rows = [key_of[k] for k in kk if k in key_of]
    m = manifest.iloc[rows].copy()
    m["usable"] = np.array(uu[:len(rows)], bool)
    log(f"clips converted: {len(m)} | usable: {int(m.usable.sum())} "
        f"({100*m.usable.mean():.1f}%) | unusable: {int((~m.usable).sum())}")
    log("usable per split: " + str(m[m.usable].split.value_counts().to_dict()))
    bad_cls = m.groupby("class").usable.mean().sort_values().head(10)
    log("classes with the lowest usable fraction:\n" + bad_cls.to_string())
    return m


# --------------------------------------------------------------------------
# archive mode: one resumable download of the whole archive (only if streaming is refused)
# --------------------------------------------------------------------------
def fetch_archive(url, dest, expected_size=None, retries=8, log=print):
    import requests
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    if dest.exists() and (not expected_size or dest.stat().st_size == expected_size):
        log(f"archive already present: {dest} ({dest.stat().st_size/2**30:.2f} GB)")
        return dest
    last_report = 0
    for attempt in range(1, retries + 1):
        existing = part.stat().st_size if part.exists() else 0
        headers = {"Range": f"bytes={existing}-"} if existing else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as r:
                if r.status_code in (401, 403):
                    raise sp_core.AuthError(f"portal returned HTTP {r.status_code} (key rejected?)")
                if r.status_code == 416 and expected_size and existing == expected_size:
                    break
                if r.status_code not in (200, 206):
                    log(f"attempt {attempt}: HTTP {r.status_code}")
                    time.sleep(min(2 ** attempt, 30))
                    continue
                mode = "ab"
                if existing and r.status_code == 200:         # server ignored Range: start over
                    log("server ignored the resume request - restarting from byte 0")
                    existing, mode = 0, "wb"
                elif existing:
                    cr = r.headers.get("Content-Range", "")
                    if not cr.startswith(f"bytes {existing}-"):
                        log(f"unexpected Content-Range {cr!r} - restarting from byte 0")
                        existing, mode = 0, "wb"
                if not existing:
                    mode = "wb"
                done = existing
                with open(part, mode) as fh:
                    for blk in r.iter_content(chunk_size=8 << 20):
                        if blk:
                            fh.write(blk)
                            done += len(blk)
                            if done - last_report >= (512 << 20):
                                last_report = done
                                tot = f" / {expected_size/2**30:.2f} GB ({100*done/expected_size:.1f}%)" if expected_size else ""
                                log(f"  downloaded {done/2**30:.2f} GB{tot}")
            break
        except sp_core.AuthError:
            raise
        except Exception as e:
            log(f"attempt {attempt}: {e.__class__.__name__}: {sp_core.redact(e)[:120]}")
            time.sleep(min(2 ** attempt, 30))
    got = part.stat().st_size if part.exists() else 0
    if expected_size and got != expected_size:
        raise RuntimeError(f"download incomplete: {got:,} of {expected_size:,} bytes. "
                           "Run this cell again to resume.")
    os.replace(part, dest)
    log(f"archive saved: {dest} ({dest.stat().st_size/2**30:.2f} GB)")
    return dest


In [ ]:
# ============================ 6. ACCESS CHECK ============================
# Tests the portal and says exactly what is wrong if something is. Run it every session.
importlib.invalidate_caches()
import sp_core, sp_features, sp_run
for m in (sp_core, sp_features, sp_run):
    importlib.reload(m)

if HAVE_ARCHIVE:
    sp_core.check_archive(ARCHIVE_PATH)                       # clear message if the file is incomplete
    MODE, URL = "archive", ""
    access = dict(mode="archive", size=os.path.getsize(ARCHIVE_PATH))
    print("\nUsing your local archive - the portal and the key are not needed.")
else:
    KEY = sp_core.sanitize_key(KEY)
    sp_core.register_secret(KEY)
    URL = URL_TEMPLATE.format(key=KEY)
    access = sp_core.preflight(URL_TEMPLATE, KEY)
    if not access["mode"]:
        raise SystemExit("STOP - fix the problem above, then rerun cells 2 and 6.\n" + access["reason"])
    MODE = access["mode"] if ACCESS_MODE == "auto" else ACCESS_MODE
    if MODE == "stream" and access["mode"] != "stream":
        raise SystemExit("ACCESS_MODE='stream' but the portal refuses range requests. Use 'auto' or 'archive'.")
print("access mode in use:", MODE)

In [ ]:
# ============================ 7. DOWNLOAD ARCHIVE (only needed in archive mode) ============================
# Resumable. Saves to Drive once; later sessions reuse it. Safe to stop and rerun.
if MODE != "archive":
    print("Not needed - videos are streamed. Skip this cell.")
elif HAVE_ARCHIVE:
    print("Not needed - you already have the archive. Skip this cell.")
else:
    sp_run.fetch_archive(URL, ARCHIVE_PATH, access["size"])

In [ ]:
# ============================ 8. INITIALISE ============================
SOURCE = URL if MODE == "stream" else ARCHIVE_PATH
if MODE == "archive" and not os.path.exists(SOURCE):
    raise SystemExit("Archive mode: run cell 7 first (the archive is not downloaded yet).")

workers = WORKERS or os.cpu_count() or 2
if not DRY_RUN:
    sp_core.download_models(MODEL_DIR, POSE_MODEL)

cfg = sp_run.make_cfg(N_FRAMES, FACE_EVERY_K, POSE_MODEL, MAX_SIDE, MODEL_DIR, TMP_DIR,
                      detector="stub" if DRY_RUN else "mediapipe", mediapipe_version=MP_VERSION)
sp_run.check_or_write_config(META, cfg)
manifest, SHA = sp_run.load_or_build_manifest(META, SOURCE, CHUNK_SIZE)
print("settings fingerprint:", cfg["fingerprint"], "| workers:", workers, "| mode:", MODE)
_ = sp_run.status(manifest, CHUNKS)

In [ ]:
# ============================ 9. BENCHMARK (run once, before the long extraction) ============================
# Measures the real cost on THIS machine for 6 clips and shows how much the sparse-face
# shortcut changes the features. Decide FACE_EVERY_K from this BEFORE the first real chunk is saved.
RUN_BENCHMARK = not DRY_RUN
if RUN_BENCHMARK:
    zf = sp_core.open_source(SOURCE)
    sample = manifest.sample(6, random_state=0).path.tolist()
    det = sp_core.Detectors(MODEL_DIR, POSE_MODEL)
    bench = sp_core.benchmark(zf, sample, cfg, det, total_clips=len(manifest), workers=workers)
    det.close(); zf.close()

In [ ]:
# ============================ 10. RUN EXTRACTION (resumable - safe to stop and rerun) ============================
summary = sp_run.run_extraction(
    manifest, SHA, cfg, SOURCE, KEY, CHUNKS, LOCAL_CHUNKS,
    machine_index=MACHINE_INDEX, machine_count=MACHINE_COUNT, workers=workers,
    max_hours=MAX_HOURS, limit_chunks=LIMIT_CHUNKS, order=ORDER)
print(summary)

In [ ]:
# ============================ 11. STATUS + INTEGRITY CHECK ============================
_ = sp_run.status(manifest, CHUNKS)
bad, wrong = sp_run.verify_chunks(CHUNKS, SHA, cfg["fingerprint"])   # deletes corrupt chunk files so they are redone

In [ ]:
# ============================ 12. BUILD FEATURES (fast, from the saved landmarks) ============================
# Safe to run on a partial dataset; rerun later to convert only the new chunks.
sp_run.build_features(CHUNKS, FEATS, cfg)
info = sp_run.feature_summary(manifest, FEATS)

## Using a second machine (old laptop)
1. Python 3.11 or 3.12 (newer versions may have no MediaPipe wheel): `pip install jupyter numpy pandas requests opencv-python-headless mediapipe==<version printed by cell 2 on Colab>`.
2. `export FDMSE_ISL_KEY=...`, start Jupyter, open this notebook. Set `MACHINE_COUNT = 2` and `MACHINE_INDEX = 1` (and `0` on Colab) so the two machines never do the same chunk.
3. Copy `meta/` from Drive into the laptop's `signsage_fdmse_v2/meta/` first (it carries the locked settings). The manifest hash printed in cell 8 **must match** Colab's.
4. When done, copy the laptop's `chunks/` files into the Drive `chunks/` folder, then run cells 11-12 on Colab.
5. In archive mode the laptop also needs the archive file: copy it from Drive to `signsage_fdmse_v2/archive/FDMSE-ISL.zip`.

## Credit and permission
Dataset: FDMSE-ISL, used with the authors' permission for the SignSage project. Keep their written reply with the project.

```
@article{PatraSuvajitPAA25,
  title={Hierarchical Windowed Graph Attention Transformer Encoder and a Large Scale Dataset for Indian Sign Language Recognition},
  author={Suvajit Patra and Arkadip Maitra and Megha Tiwari and K. Kumaran and Swathy Prabhu and Swami Punyeshwarananda and Soumitra Samanta},
  journal={Pattern Analysis and Applications}, volume={28}, number={3}, pages={148}, year={2025}
}
```
The feature code (`sp_features.py`, sign_features_v3) is carried over unchanged from the SignSage v1 backend.